# Analýza nároku na výdaje

Tento poznámkový blok ukazuje, jak vytvořit agenty, kteří používají pluginy k zpracování cestovních výdajů z místních obrázků účtenek, generují e-mail s nárokem na výdaje a vizualizují data o výdajích pomocí koláčového grafu. Agenti dynamicky volí funkce na základě kontextu úkolu.

Kroky:
1. OCR agent zpracuje místní obrázek účtenky a extrahuje data o cestovních výdajích.
2. E-mailový agent vygeneruje e-mail s nárokem na výdaje.

### Příklad scénáře cestovních výdajů:
Představte si, že jste zaměstnanec cestující na obchodní schůzku do jiného města. Vaše společnost má politiku proplácení všech rozumných cestovních nákladů. Zde je rozpis možných cestovních výdajů:
- Doprava:
Letecká doprava tam a zpět z vašeho domovského města do cílového města.
Taxi nebo služby jízdy na zavolání na letiště a z letiště.
Místní doprava v cílovém městě (jako je veřejná doprava, pronájem aut nebo taxi).

- Ubytování:
Pobyt v hotelu na tři noci v středně drahém obchodním hotelu blízko místa schůzky.

- Jídlo:
Denní stravné na snídani, oběd a večeři podle firemních pravidel.

- Různé výdaje:
Poplatky za parkování na letišti.
Poplatky za přístup k internetu v hotelu.
Spropitné nebo malé servisní poplatky.

- Dokumentace:
Odevzdáte všechny účtenky (letenky, taxi, hotel, jídlo atd.) a vyplněnou zprávu o výdajích k proplacení.


## Import požadovaných knihoven

Importujte nezbytné knihovny a moduly pro tento notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Definujte modely výdajů

 Vytvořte model Pydantic pro jednotlivé výdaje a třídu ExpenseFormatter pro převod dotazu uživatele do strukturovaných dat o výdajích.

 Každý výdaj bude reprezentován ve formátu:
 `{'date': '07-Mar-2025', 'description': 'let do destinace', 'amount': 675.99, 'category': 'Doprava'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definování nástrojů - Generování e-mailu

Vytvořte funkci nástroje pro vygenerování e-mailu k podání žádosti o náhradu výdajů.
- Tento nástroj používá dekorátor `@tool` z Microsoft Agent Framework.
- Vypočítá celkovou částku výdajů a naformátuje podrobnosti do textu e-mailu.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Nástroj pro extrakci cestovních výdajů z obrázků účtenek

Vytvořte funkci nástroje pro extrakci cestovních výdajů z obrázků účtenek.
- Tento nástroj používá dekorátor `@tool` z Microsoft Agent Framework.
- Čte obrázek účtenky, kóduje jej do base64 a vrací data URI, aby jej agent mohl analyzovat.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Zpracování výdajů

Definujte agenty a propojte je do sekvenčního pracovního postupu pomocí `WorkflowBuilder`.
- OCR agent extrahuje strukturovaná data o výdajích z obrázku účtenky pomocí nástroje `load_receipt_image`.
- Email agent vezme extrahovaná data a vygeneruje profesionální e-mail s žádostí o náhradu výdajů pomocí nástroje `generate_expense_email`.
- `WorkflowBuilder` s `add_edge` vytvoří sekvenční pipeline: OCR Agent → Email Agent.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hlavní funkce

Sestavte sekvenční pracovní postup a spusťte jej ke zpracování obrázku účtenky a vytvoření e-mailu s nárokem na náklady.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:  
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). I když usilujeme o přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za závazný zdroj. Pro důležité informace se doporučuje profesionální lidský překlad. Nejsme odpovědni za jakákoliv nedorozumění nebo nesprávné interpretace vyplývající z užití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
